In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def save_overall_runtime_plot(metrics: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 6))
    x = range(len(metrics))
    width = 0.25

    plt.bar([i - width for i in x], metrics["train_time_sec"], width=width, label="Train Time (s)")
    plt.bar(x, metrics["predict_time_sec"], width=width, label="Predict Time (s)")
    plt.bar([i + width for i in x], metrics["split_eval_sec"], width=width, label="Split Eval (s)")

    plt.xticks(list(x), metrics["dataset"], rotation=15)
    plt.ylabel("Time (seconds)")
    plt.title("Runtime Comparison Across Datasets")
    plt.legend()
    plt.tight_layout()
    plt.savefig("runtime_comparison.png", dpi=180)
    plt.close()


def save_accuracy_plot(metrics: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 6))
    x = range(len(metrics))
    width = 0.35

    plt.bar([i - width / 2 for i in x], metrics["train_accuracy"] * 100.0, width=width, label="Train Accuracy")
    plt.bar([i + width / 2 for i in x], metrics["test_accuracy"] * 100.0, width=width, label="Test Accuracy")

    plt.xticks(list(x), metrics["dataset"], rotation=15)
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy Comparison Across Datasets")
    plt.legend()
    plt.tight_layout()
    plt.savefig("accuracy_comparison.png", dpi=180)
    plt.close()


def save_scalability_plots(scale: pd.DataFrame) -> None:
    for ds_name, part in scale.groupby("dataset"):
        part = part.sort_values("n_samples")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

        axes[0].plot(part["n_samples"], part["train_time_sec"], marker="o", label="Train Time")
        axes[0].plot(part["n_samples"], part["predict_time_sec"], marker="s", label="Predict Time")
        axes[0].set_title(f"Scalability Runtime - {ds_name}")
        axes[0].set_xlabel("Number of Samples")
        axes[0].set_ylabel("Time (seconds)")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].plot(part["n_samples"], part["test_accuracy"] * 100.0, marker="^", color="tab:green")
        axes[1].set_title(f"Scalability Accuracy - {ds_name}")
        axes[1].set_xlabel("Number of Samples")
        axes[1].set_ylabel("Test Accuracy (%)")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        safe_name = ds_name.lower().replace(" ", "_")
        plt.savefig(f"scalability_{safe_name}.png", dpi=180)
        plt.close()


def write_scalability_comments(scale: pd.DataFrame) -> None:
    lines = ["Scalability comments based on benchmark_scalability.csv", ""]

    for ds_name, part in scale.groupby("dataset"):
        part = part.sort_values("n_samples")
        first = part.iloc[0]
        last = part.iloc[-1]

        size_ratio = float(last["n_samples"]) / max(1.0, float(first["n_samples"]))
        train_ratio = float(last["train_time_sec"]) / max(1e-9, float(first["train_time_sec"]))
        pred_ratio = float(last["predict_time_sec"]) / max(1e-9, float(first["predict_time_sec"]))
        acc_delta_pp = (float(last["test_accuracy"]) - float(first["test_accuracy"])) * 100.0

        lines.append(f"Dataset: {ds_name}")
        lines.append(f"- Size scaling: x{size_ratio:.3f}")
        lines.append(f"- Train-time scaling: x{train_ratio:.3f}")
        lines.append(f"- Inference-time scaling: x{pred_ratio:.3f}")
        lines.append(f"- Test-accuracy change: {acc_delta_pp:.3f} percentage points")

        if train_ratio <= size_ratio * 1.2:
            lines.append("- Comment: runtime appears close to linear with dataset growth.")
        else:
            lines.append("- Comment: runtime grows faster than linear; check depth/bin settings.")

        if acc_delta_pp >= 0.5:
            lines.append("- Comment: larger sample sizes improved generalization noticeably.")
        elif acc_delta_pp <= -0.5:
            lines.append("- Comment: larger sample sizes reduced test accuracy; inspect label distribution and class balance.")
        else:
            lines.append("- Comment: test accuracy is relatively stable across sample sizes.")

        lines.append("")

    with open("scalability_comments.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def main() -> None:
    metrics = pd.read_csv("benchmark_metrics.csv")
    scale = pd.read_csv("benchmark_scalability.csv")

    save_overall_runtime_plot(metrics)
    save_accuracy_plot(metrics)
    save_scalability_plots(scale)
    write_scalability_comments(scale)

    print("Generated:")
    print("- runtime_comparison.png")
    print("- accuracy_comparison.png")
    print("- scalability_<dataset>.png for each dataset")
    print("- scalability_comments.txt")


if __name__ == "__main__":
    main()


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


DATASETS = [
    {"name": "Iris", "file": "Iris.csv", "has_header": True},
    {"name": "Shuttle", "file": "shuttle.csv", "has_header": True},
    {"name": "LetterRecognition", "file": "letter-recognition.csv", "has_header": True},
    {"name": "Skin_NonSkin", "file": "Skin_NonSkin.csv", "has_header": True},
    {"name": "Synthetic_dataset_1000k_200f", "file": "synthetic_1000k_200f.csv", "has_header": True},
]

FRACTIONS = [0.10, 0.25, 0.50, 0.75, 1.00]
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Match milestone benchmarking style (fixed params for all datasets)
SK_PARAMS = {
    "criterion": "gini",
    "max_depth": 8,
    "min_samples_split": 2,
    "random_state": RANDOM_STATE,
}


def infer_label_column(df: pd.DataFrame) -> int:
    first = pd.to_numeric(df.iloc[:, 0], errors="coerce")
    last = pd.to_numeric(df.iloc[:, -1], errors="coerce")

    first_numeric_ratio = first.notna().mean()
    last_numeric_ratio = last.notna().mean()

    # Letter dataset style: first column is categorical label.
    if first_numeric_ratio < 0.95 and last_numeric_ratio >= 0.95:
        return 0
    return df.shape[1] - 1


def load_dataset(spec: dict) -> tuple[pd.DataFrame, pd.Series]:
    path = Path(spec["file"])
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset file: {path.resolve()}")

    header = 0 if spec["has_header"] else None
    df = pd.read_csv(path, header=header, low_memory=False, skipinitialspace=True)

    label_col = infer_label_column(df)
    y = df.iloc[:, label_col]
    X = df.drop(df.columns[label_col], axis=1)

    # Ensure all features numeric for sklearn tree.
    X = X.apply(pd.to_numeric, errors="coerce")
    if X.isna().any().any():
        raise ValueError(f"Non-numeric feature values detected in {spec['name']} after parsing")

    # Encode labels if categorical.
    if not pd.api.types.is_numeric_dtype(y):
        y = y.astype("category").cat.codes

    return X, y


def benchmark_sklearn_once(dataset_name: str, X: pd.DataFrame, y: pd.Series) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = DecisionTreeClassifier(**SK_PARAMS)

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    t1 = time.perf_counter()

    train_preds = model.predict(X_train)

    p0 = time.perf_counter()
    test_preds = model.predict(X_test)
    p1 = time.perf_counter()

    return {
        "dataset": dataset_name,
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_classes": int(pd.Series(y).nunique()),
        "train_samples": int(X_train.shape[0]),
        "test_samples": int(X_test.shape[0]),
        "max_depth": SK_PARAMS["max_depth"],
        "min_samples_split": SK_PARAMS["min_samples_split"],
        "bin_count": 8,  # placeholder to align schema with sequential CSV
        "train_time_sec": float(t1 - t0),
        "split_eval_sec": np.nan,  # not exposed by sklearn internals
        "tree_overhead_sec": np.nan,
        "predict_time_sec": float(p1 - p0),
        "infer_time_per_sample_us": float((p1 - p0) * 1e6 / max(1, X_test.shape[0])),
        "train_accuracy": float((train_preds == y_train).mean()),
        "test_accuracy": float((test_preds == y_test).mean()),
    }


def run_sklearn_benchmarks() -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    scalability_rows = []

    for spec in DATASETS:
        X_full, y_full = load_dataset(spec)
        n_total = len(X_full)

        full_metrics = benchmark_sklearn_once(spec["name"], X_full, y_full)
        metric_rows.append(full_metrics)

        for frac in FRACTIONS:
            n_sub = int(round(frac * n_total))
            n_sub = min(max(200, n_sub), n_total)

            idx = np.random.RandomState(RANDOM_STATE + int(frac * 1000)).choice(
                n_total, size=n_sub, replace=False
            )
            X_sub = X_full.iloc[idx].reset_index(drop=True)
            y_sub = y_full.iloc[idx].reset_index(drop=True)

            m = benchmark_sklearn_once(spec["name"], X_sub, y_sub)
            scalability_rows.append(
                {
                    "dataset": spec["name"],
                    "fraction": float(frac),
                    "n_samples": int(m["n_samples"]),
                    "train_time_sec": float(m["train_time_sec"]),
                    "predict_time_sec": float(m["predict_time_sec"]),
                    "test_accuracy": float(m["test_accuracy"]),
                }
            )

    metrics_df = pd.DataFrame(metric_rows)
    scale_df = pd.DataFrame(scalability_rows)
    return metrics_df, scale_df


def save_sklearn_plots(metrics: pd.DataFrame, scale: pd.DataFrame) -> None:
    # Overall runtime plot
    plt.figure(figsize=(10, 6))
    x = np.arange(len(metrics))
    width = 0.35
    plt.bar(x - width / 2, metrics["train_time_sec"], width=width, label="Train Time (s)")
    plt.bar(x + width / 2, metrics["predict_time_sec"], width=width, label="Predict Time (s)")
    plt.xticks(x, metrics["dataset"], rotation=15)
    plt.ylabel("Time (seconds)")
    plt.title("Sklearn Runtime Comparison Across Datasets")
    plt.legend()
    plt.tight_layout()
    plt.savefig("runtime_comparison_sklearn.png", dpi=180)
    plt.close()

    # Accuracy plot
    plt.figure(figsize=(10, 6))
    plt.bar(x - width / 2, metrics["train_accuracy"] * 100.0, width=width, label="Train Accuracy")
    plt.bar(x + width / 2, metrics["test_accuracy"] * 100.0, width=width, label="Test Accuracy")
    plt.xticks(x, metrics["dataset"], rotation=15)
    plt.ylabel("Accuracy (%)")
    plt.title("Sklearn Accuracy Comparison Across Datasets")
    plt.legend()
    plt.tight_layout()
    plt.savefig("accuracy_comparison_sklearn.png", dpi=180)
    plt.close()

    # Per-dataset scalability plots
    for ds_name, part in scale.groupby("dataset"):
        part = part.sort_values("n_samples")
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

        axes[0].plot(part["n_samples"], part["train_time_sec"], marker="o", label="Train Time")
        axes[0].plot(part["n_samples"], part["predict_time_sec"], marker="s", label="Predict Time")
        axes[0].set_title(f"Sklearn Scalability Runtime - {ds_name}")
        axes[0].set_xlabel("Number of Samples")
        axes[0].set_ylabel("Time (seconds)")
        axes[0].grid(alpha=0.3)
        axes[0].legend()

        axes[1].plot(part["n_samples"], part["test_accuracy"] * 100.0, marker="^", color="tab:green")
        axes[1].set_title(f"Sklearn Scalability Accuracy - {ds_name}")
        axes[1].set_xlabel("Number of Samples")
        axes[1].set_ylabel("Test Accuracy (%)")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        safe_name = ds_name.lower().replace(" ", "_")
        plt.savefig(f"scalability_sklearn_{safe_name}.png", dpi=180)
        plt.close()


def save_comparison_plots(seq_metrics: pd.DataFrame, sk_metrics: pd.DataFrame) -> None:
    merged = seq_metrics.merge(sk_metrics, on="dataset", suffixes=("_seq", "_sk"))
    x = np.arange(len(merged))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width / 2, merged["train_time_sec_seq"], width=width, label="Sequential")
    plt.bar(x + width / 2, merged["train_time_sec_sk"], width=width, label="Sklearn")
    plt.xticks(x, merged["dataset"], rotation=15)
    plt.ylabel("Train Time (seconds)")
    plt.title("Train Time Comparison: Sequential vs Sklearn")
    plt.legend()
    plt.tight_layout()
    plt.savefig("comparison_train_time_seq_vs_sklearn.png", dpi=180)
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.bar(x - width / 2, merged["test_accuracy_seq"] * 100.0, width=width, label="Sequential")
    plt.bar(x + width / 2, merged["test_accuracy_sk"] * 100.0, width=width, label="Sklearn")
    plt.xticks(x, merged["dataset"], rotation=15)
    plt.ylabel("Test Accuracy (%)")
    plt.title("Test Accuracy Comparison: Sequential vs Sklearn")
    plt.legend()
    plt.tight_layout()
    plt.savefig("comparison_test_accuracy_seq_vs_sklearn.png", dpi=180)
    plt.close()


# Run sklearn benchmarks and generate outputs
sk_metrics, sk_scale = run_sklearn_benchmarks()
sk_metrics.to_csv("benchmark_metrics_sklearn.csv", index=False)
sk_scale.to_csv("benchmark_scalability_sklearn.csv", index=False)

save_sklearn_plots(sk_metrics, sk_scale)

seq_metrics = pd.read_csv("benchmark_metrics.csv")
save_comparison_plots(seq_metrics, sk_metrics)

print("Generated sklearn artifacts:")
print("- benchmark_metrics_sklearn.csv")
print("- benchmark_scalability_sklearn.csv")
print("- runtime_comparison_sklearn.png")
print("- accuracy_comparison_sklearn.png")
print("- scalability_sklearn_<dataset>.png")
print("- comparison_train_time_seq_vs_sklearn.png")
print("- comparison_test_accuracy_seq_vs_sklearn.png")